Load Lama model

In [1]:
import os, sys, torch
from omegaconf import OmegaConf

# Set LaMa source code path
LAMA_PATH = "/gpfs/work/bio/yixuanli2204/FYP_Project/code/lama"
if LAMA_PATH not in sys.path:
    sys.path.append(LAMA_PATH)

from saicinpainting.training.trainers import load_checkpoint
from saicinpainting.evaluation.utils import move_to_device

# Loading weights and configurations
device = torch.device("cuda")
model_dir = "/gpfs/work/bio/yixuanli2204/FYP_Project/weights/lama"
config_path = os.path.join(model_dir, "config.yaml") 
model_path = os.path.join(model_dir, "bigLama.ckpt")
train_config = OmegaConf.load(config_path)
train_config.training_model.predict_only = True 

# Initialize the model

model = load_checkpoint(train_config, model_path, strict=False, map_location='cpu')
model.to(device)
model.eval()
print("LaMa loaded successfully")

/gpfs/work/bio/yixuanli2204/.conda/envs/fyp/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Detectron v2 is not installed
LaMa loaded successfully


test 5 images

In [3]:
import os, sys, cv2, torch
import numpy as np
from saicinpainting.evaluation.utils import move_to_device
from omegaconf import OmegaConf

IMG_DIR = "/gpfs/work/bio/yixuanli2204/FYP_Project/data/gold_real_d2/dermoscopic_image"
MASK_DIR = "/gpfs/work/bio/yixuanli2204/FYP_Project/data/gold_real_d2/sam_predicted_masks"
OUT_DIR = "/gpfs/work/bio/yixuanli2204/FYP_Project/data/gold_real_d2/restored_test_micro_dilate"

os.makedirs(OUT_DIR, exist_ok=True)


all_files = sorted([f for f in os.listdir(IMG_DIR) if f.lower().endswith('.png')])[:5] #


for i, fname in enumerate(all_files):
    img_path = os.path.join(IMG_DIR, fname)
    mask_path = os.path.join(MASK_DIR, fname)
    out_path = os.path.join(OUT_DIR, f"micro_dilate_{fname}")
    
    try:
        img = cv2.imread(img_path)
        mask = cv2.imread(mask_path, 0)
        
        if img is None or mask is None: continue

        # 1. Size processing (512x512)
        h_orig, w_orig = img.shape[:2]
        img_res = cv2.resize(cv2.cvtColor(img, cv2.COLOR_BGR2RGB), (512, 512))
        mask_res = cv2.resize(mask, (512, 512))

        # 2. 3x3 slight expansion. Expand outward by approximately 1 pixel to remove residual color blocks along the edges of the hair.
        _, mask_bin = cv2.threshold(mask_res, 127, 255, cv2.THRESH_BINARY)
        kernel = np.ones((3, 3), np.uint8) 
        mask_final = cv2.dilate(mask_bin, kernel, iterations=1)

        # 3. Structural Inference Batch
        batch = {
            "image": torch.from_numpy(img_res).float().permute(2,0,1).unsqueeze(0) / 255.0,
            "mask": torch.from_numpy(mask_final).unsqueeze(0).unsqueeze(0).float() / 255.0
        }
        batch = move_to_device(batch, device) 
        
        # 4. Model Inference
        with torch.no_grad():
            output = model(batch)
            res = output["inpainted"][0].permute(1, 2, 0).cpu().numpy()
            res = (np.clip(res, 0, 1) * 255).astype(np.uint8)
            res = cv2.cvtColor(res, cv2.COLOR_RGB2BGR)

        # 5. Resize and save
        cv2.imwrite(out_path, cv2.resize(res, (w_orig, h_orig)))
        print(f"finish: [{i+1}/5]")

    except Exception as e:
        print(f"❌ Failed to process {fname}: {str(e)}")

print(f" Fine-tuning test complete! Please check {OUT_DIR}")

finish: [1/5]
finish: [2/5]
finish: [3/5]
finish: [4/5]
finish: [5/5]
 Fine-tuning test complete! Please check /gpfs/work/bio/yixuanli2204/FYP_Project/data/gold_real_d2/restored_test_micro_dilate


Batch Repair

In [ ]:
import os, sys, cv2, torch
import numpy as np
from saicinpainting.evaluation.utils import move_to_device

# Configure paths
IMG_DIR = "/gpfs/work/bio/yixuanli2204/FYP_Project/data/gold_real_d2/dermoscopic_image"
MASK_DIR = "/gpfs/work/bio/yixuanli2204/FYP_Project/data/gold_real_d2/sam_predicted_masks"
OUT_DIR = "/gpfs/work/bio/yixuanli2204/FYP_Project/data/gold_real_d2/restored_image_final" # Final output directory

os.makedirs(OUT_DIR, exist_ok=True)

#  Batch processing logic 
print(f"Starting full 500-image restoration (3x3 micro-dilation mode)...")

# Get all image files
all_files = sorted([f for f in os.listdir(IMG_DIR) if f.lower().endswith('.png')])
total = len(all_files)

# Define 3x3 dilation kernel
kernel = np.ones((3, 3), np.uint8)

for i, fname in enumerate(all_files):
    img_path = os.path.join(IMG_DIR, fname)
    mask_path = os.path.join(MASK_DIR, fname)
    out_path = os.path.join(OUT_DIR, fname)
    
    # Skip if file already exists (resume from breakpoint)
    if os.path.exists(out_path):
        continue

    try:
        # Read original image and SAM-generated mask
        img = cv2.imread(img_path)
        mask = cv2.imread(mask_path, 0)
        
        if img is None or mask is None:
            print(f"Skipping {fname}: File reading failed")
            continue

        h_orig, w_orig = img.shape[:2]
        
        # Preprocessing: 512 size is optimal for LaMa
        img_res = cv2.resize(cv2.cvtColor(img, cv2.COLOR_BGR2RGB), (512, 512))
        mask_res = cv2.resize(mask, (512, 512))

        # Apply 3x3 micro-dilation
        _, mask_bin = cv2.threshold(mask_res, 127, 255, cv2.THRESH_BINARY)
        mask_final = cv2.dilate(mask_bin, kernel, iterations=1)

        # Construct Tensor
        batch = {
            "image": torch.from_numpy(img_res).float().permute(2,0,1).unsqueeze(0) / 255.0,
            "mask": torch.from_numpy(mask_final).unsqueeze(0).unsqueeze(0).float() / 255.0
        }
        batch = move_to_device(batch, device)
        
        # Model inference
        with torch.no_grad():
            output = model(batch)
            res = output["inpainted"][0].permute(1, 2, 0).cpu().numpy()
            res = (np.clip(res, 0, 1) * 255).astype(np.uint8)
            res = cv2.cvtColor(res, cv2.COLOR_RGB2BGR)

        # Restore original dimensions and save
        cv2.imwrite(out_path, cv2.resize(res, (w_orig, h_orig)))
        
        if (i + 1) % 50 == 0:
            print(f"Completed: [{i+1}/{total}]")

    except Exception as e:
        print(f"Failed to process {fname}: {str(e)}")

print(f"All 500 images have been successfully restored! Output directory: {OUT_DIR}")